#Despliegue

-cargar modelo

-cargar datos futuros

-preparar datos futuros

-aplicar el modeolo

In [1]:
#Cargamos librerías principales

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
import pickle
filename = 'modelo-class.pkl'

# Cargar el contenido del archivo pickle en una sola variable para inspección
#loaded_content = pickle.load(open(filename, 'rb'))
modelo, encoder, variables, scaler = pickle.load(open(filename, 'rb'))

In [ ]:
#print(type(loaded_content))

<class 'list'>


In [ ]:
#print(loaded_content)

[DecisionTreeClassifier(max_depth=10, min_samples_leaf=50), LabelEncoder(), array(['age', 'avg_glucose_level', "smoking_status_'formerly smoked'",
       "smoking_status_'never smoked'", 'smoking_status_Unknown',
       'smoking_status_smokes', 'hypertension_Yes', 'heart_disease_Yes',
       'ever_married_Yes'], dtype=object), MinMaxScaler()]


In [3]:
#Cargamos los datos futuros
data = pd.read_excel("ataque_corazon.xlsx")
data.head()

,age,hypertension,heart_disease,ever_married,avg_glucose_level,smoking_status,stroke_ataque_corazon
0,1,No,No,No,122.04,Unknown,No
1,79,No,No,Yes,79.03,Unknown,No
2,40,No,No,Yes,60.96,'never smoked',No
3,4,No,No,No,107.25,Unknown,No
4,8,No,No,No,106.51,Unknown,No


In [ ]:
#Interfaz gráfica
#Se crea interfaz gráfica con streamlit para captura de los datos
#STREAMLIT SOLO SE EJECUTA EN UN SERVIDOR WEB

import streamlit as st

st.title('Predicción de Ataque al corazon')

age = st.slider('Edad', min_value=0, max_value=100, value=30, step=1)
hypertension = st.selectbox('Hipertensión', ['Yes', 'No'])
heart_disease = st.selectbox('Enfermedad Cardíaca', ['Yes', 'No'])
ever_married = st.selectbox('¿Alguna vez casado?', ['Yes', 'No'])
avg_glucose_level = st.number_input('Nivel de glucosa', value=80.0)
smoking_status = st.selectbox('Estado de fumador', ['formerly smoked', 'never smoked', 'smokes', 'Unknown'])


#Dataframe
data = pd.DataFrame([{
    'age': age,
    'hypertension': hypertension,
    'heart_disease': heart_disease,
    'ever_married': ever_married,
    'avg_glucose_level': avg_glucose_level,
    'smoking_status': smoking_status
}])
#datos = [[Edad, videojuego,Plataforma,Sexo,Consumidor_habitual]]
#data = pd.DataFrame(datos, columns=['Edad', 'videojuego','Plataforma','Sexo','Consumidor_habitual']) #Dataframe con los mismos nombres de variables


In [4]:
#Se realiza la preparación
data_preparada=data.copy()

#En despliegue drop_first= False

data_preparada = pd.get_dummies(data_preparada, columns=['smoking_status'], drop_first=False, dtype=int)
data_preparada = pd.get_dummies(data_preparada, columns=['hypertension','heart_disease','ever_married',], drop_first=True, dtype=int)
data_preparada.head()

,age,avg_glucose_level,stroke_ataque_corazon,smoking_status_'formerly smoked',smoking_status_'never smoked',smoking_status_Unknown,smoking_status_smokes,hypertension_Yes,heart_disease_Yes,ever_married_Yes
0,1,122.04,No,0,0,1,0,0,0,0
1,79,79.03,No,0,0,1,0,0,0,1
2,40,60.96,No,0,1,0,0,0,0,1
3,4,107.25,No,0,0,1,0,0,0,0
4,8,106.51,No,0,0,1,0,0,0,0


In [5]:
from numpy._core.fromnumeric import var
#Se adicionan las columnas faltantes
#variables=['age', 'hypertension', 'heart_disease', 'ever_married', 'avg_glucose_level', 'smoking_status', 'stroke_ataque_corazon']
data_preparada=data_preparada.reindex(columns=variables,fill_value=0)
data_preparada.head()

,age,avg_glucose_level,smoking_status_'formerly smoked',smoking_status_'never smoked',smoking_status_Unknown,smoking_status_smokes,hypertension_Yes,heart_disease_Yes,ever_married_Yes
0,1,122.04,0,0,1,0,0,0,0
1,79,79.03,0,0,1,0,0,0,1
2,40,60.96,0,1,0,0,0,0,1
3,4,107.25,0,0,1,0,0,0,0
4,8,106.51,0,0,1,0,0,0,0


In [ ]:
#Se normaliza la edad para predecir con Knn, Red, SVM
#En los despliegues no se llama fit
#data_preparada[['Edad']]= min_max_scaler.transform(data_preparada[['Edad']])
#data_preparada.head()

#**Predicciones**

In [6]:
#Hacemos la predicción con el Tree
Y_pred = modelo.predict(data_preparada)
print(Y_pred)

[0 0 0 ... 0 0 0]


In [7]:
data['Prediccion']=Y_pred
data.head()

,age,hypertension,heart_disease,ever_married,avg_glucose_level,smoking_status,stroke_ataque_corazon,Prediccion
0,1,No,No,No,122.04,Unknown,No,0
1,79,No,No,Yes,79.03,Unknown,No,0
2,40,No,No,Yes,60.96,'never smoked',No,0
3,4,No,No,No,107.25,Unknown,No,0
4,8,No,No,No,106.51,Unknown,No,0


In [8]:
#Predicciones finales
data

,age,hypertension,heart_disease,ever_married,avg_glucose_level,smoking_status,stroke_ataque_corazon,Prediccion
0,1,No,No,No,122.04,Unknown,No,0
1,79,No,No,Yes,79.03,Unknown,No,0
2,40,No,No,Yes,60.96,'never smoked',No,0
3,4,No,No,No,107.25,Unknown,No,0
4,8,No,No,No,106.51,Unknown,No,0
...,...,...,...,...,...,...,...,...
5105,79,Yes,No,Yes,92.43,'never smoked',No,0
5106,80,No,No,Yes,110.66,Unknown,Yes,0
5107,80,No,No,Yes,84.86,Unknown,No,0
5108,80,Yes,No,Yes,83.75,'never smoked',No,0


In [ ]:
# Recordar medida de error del modelo

st.warning("El modelo tiene un error del 12%")